In [ ]:
from ultralytics import YOLO

# Cargar modelo preentrenado
model = YOLO("yolo11n.pt")

# Entrenar
model.train(
    data="dataset.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    name="yolo_matriculas",
    device=0
)

# Predicción en validación
results = model.predict(source='C:/Users/juanf/Desktop/Large-License-Plate-Detection-Dataset/images/val', save=True)


In [1]:
from ultralytics import YOLO
import cv2
from collections import defaultdict
import csv
from PIL import Image
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq
import numpy as np

# ============================================================================
# CONFIGURACIÓN DEL MODELO smolVLM para OCR
# ============================================================================
print("Cargando modelo smolVLM...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "HuggingFaceTB/SmolVLM-Instruct"

processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForVision2Seq.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
print(f"Modelo cargado en: {device}")

# ============================================================================
# FUNCIÓN PARA EXTRAER TEXTO DE MATRÍCULA CON smolVLM
# ============================================================================
def extraer_texto_matricula(frame, x1, y1, x2, y2):
    """
    Extrae el texto de una matrícula usando smolVLM
    """
    try:
        # Extraer ROI de la matrícula con un pequeño padding
        padding = 5
        roi = frame[max(0, y1-padding):min(frame.shape[0], y2+padding), 
                   max(0, x1-padding):min(frame.shape[1], x2+padding)]
        
        if roi.size == 0:
            return ""
        
        # Convertir de BGR (OpenCV) a RGB (PIL)
        roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(roi_rgb)
        
        # Prompt específico para lectura de matrículas
        prompt = "Read the license plate number in this image. Only output the alphanumeric characters you see, without spaces or additional text."
        
        # Preparar entrada para el modelo
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": prompt}
                ]
            }
        ]
        
        # Procesar con smolVLM
        text = processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = processor(text=[text], images=[pil_image], return_tensors="pt")
        inputs = inputs.to(device)
        
        # Generar texto
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=50,
                do_sample=False
            )
        
        # Decodificar resultado
        generated_texts = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )
        
        # Extraer solo el texto de la respuesta
        texto = generated_texts[0].split("Assistant:")[-1].strip()
        
        # Limpiar el texto: solo alfanuméricos
        texto_limpio = ''.join(c for c in texto if c.isalnum()).upper()
        
        return texto_limpio
    
    except Exception as e:
        print(f"Error al procesar matrícula: {e}")
        return ""

# ============================================================================
# FUNCIÓN PARA ASOCIAR MATRÍCULAS CON VEHÍCULOS
# ============================================================================
def asociar_matricula_con_vehiculo(plate_box, vehicle_boxes):
    """
    Encuentra el vehículo que contiene o está más cerca de la matrícula
    Retorna el índice del vehículo o None si no hay asociación
    """
    px1, py1, px2, py2 = plate_box
    plate_center_x = (px1 + px2) / 2
    plate_center_y = (py1 + py2) / 2
    
    mejor_vehiculo = None
    mejor_distancia = float('inf')
    
    for idx, vbox in enumerate(vehicle_boxes):
        vx1, vy1, vx2, vy2 = vbox
        
        # Verificar si la matrícula está dentro del vehículo
        if vx1 <= plate_center_x <= vx2 and vy1 <= plate_center_y <= vy2:
            return idx
        
        # Calcular distancia al centro del vehículo
        vcenter_x = (vx1 + vx2) / 2
        vcenter_y = (vy1 + vy2) / 2
        distancia = np.sqrt((plate_center_x - vcenter_x)**2 + (plate_center_y - vcenter_y)**2)
        
        if distancia < mejor_distancia:
            mejor_distancia = distancia
            mejor_vehiculo = idx
    
    # Solo asociar si la distancia es razonable (menos de 200 píxeles)
    if mejor_distancia < 200:
        return mejor_vehiculo
    return None

# ============================================================================
# CARGA DE MODELOS YOLO
# ============================================================================
general = YOLO("yolo11n.pt")
matriculas = YOLO("runs/detect/yolo_matriculas/weights/best.pt")

# ============================================================================
# CONFIGURACIÓN
# ============================================================================
video_path = "C0142.MP4"
output_path = "detecciones_combinadas_final.mp4"
csv_path = "detecciones_tracking_matriculas.csv"
tracker = "bytetrack.yaml"

conf_general = 0.5
conf_plate = 0.3
classes_general = [0, 2, 3, 5, 7]  # person, car, motorcycle, bus, truck

# OPTIMIZACIÓN: Procesar OCR solo cada N frames
OCR_CADA_N_FRAMES = 5  # Ajusta este valor (5 = cada 5 frames, 10 = más rápido pero menos preciso)

# ============================================================================
# INICIALIZACIÓN
# ============================================================================
writer = None
ids_por_clase = defaultdict(set)

# OPTIMIZACIÓN: Cache de matrículas por tracking ID
matriculas_cache = {}  # track_id -> {'texto': str, 'confianza': float, 'bbox': tuple}

# Crear archivo CSV
csv_file = open(csv_path, 'w', newline='', encoding='utf-8')
csv_writer = csv.writer(csv_file)
csv_writer.writerow([
    'fotograma',
    'tipo_objeto',
    'confianza',
    'identificador_tracking',
    'x1', 'y1', 'x2', 'y2',
    'tiene_matricula',
    'confianza_matricula',
    'mx1', 'my1', 'mx2', 'my2',
    'texto_matricula'
])

# ============================================================================
# PROCESAMIENTO DEL VIDEO
# ============================================================================
print("Iniciando procesamiento del video...")
print(f"Optimizaciones activas:")
print(f"  - OCR cada {OCR_CADA_N_FRAMES} frames")
print(f"  - Cache de matrículas por tracking ID")
print(f"  - Sin visualización en tiempo real")

results_stream = general.track(
    source=video_path,
    tracker=tracker,
    classes=classes_general,
    conf=conf_general,
    persist=True,
    stream=True
)

for frame_num, r in enumerate(results_stream):
    if frame_num % 10 == 0:
        print(f"Procesando fotograma {frame_num}...")
    
    # Obtener frame original
    frame = r.orig_img.copy()
    
    # Inicializar escritor de video
    if writer is None:
        h, w = frame.shape[:2]
        fps = 30
        writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    
    # Dibujar tracking
    tracked_frame = r.plot()
    
    # Recopilar información de objetos trackeados
    objetos_trackeados = []
    if hasattr(r, "boxes") and r.boxes is not None:
        for box in r.boxes:
            if box.id is None:
                continue
            
            cls = int(box.cls)
            track_id = int(box.id)
            conf = float(box.conf)
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            
            ids_por_clase[cls].add(track_id)
            
            objetos_trackeados.append({
                'cls': cls,
                'track_id': track_id,
                'conf': conf,
                'bbox': (x1, y1, x2, y2),
                'tipo': general.names[cls]
            })
    
    # OPTIMIZACIÓN: Solo detectar matrículas y hacer OCR cada N frames
    matriculas_detectadas = []
    annotated = tracked_frame.copy()
    
    if frame_num % OCR_CADA_N_FRAMES == 0:
        # Detectar matrículas
        res_plate = matriculas(frame, conf=conf_plate, verbose=False)[0]
        
        if hasattr(res_plate, "boxes") and res_plate.boxes is not None:
            for pbox in res_plate.boxes:
                x1, y1, x2, y2 = map(int, pbox.xyxy[0].tolist())
                conf_plate_det = float(pbox.conf)
                
                # Extraer texto con smolVLM
                texto_matricula = extraer_texto_matricula(frame, x1, y1, x2, y2)
                
                matriculas_detectadas.append({
                    'bbox': (x1, y1, x2, y2),
                    'conf': conf_plate_det,
                    'texto': texto_matricula
                })
                
                # Dibujar solo el rectángulo de la matrícula (sin texto)
                cv2.rectangle(annotated, (x1, y1), (x2, y2), (255, 0, 0), 2)
    else:
        # En frames intermedios, solo dibujar las matrículas del cache
        for track_id, mat_info in matriculas_cache.items():
            # Buscar si este vehículo sigue presente en el frame actual
            for obj in objetos_trackeados:
                if obj['track_id'] == track_id and obj['tipo'] in ['car', 'motorcycle', 'bus', 'truck']:
                    # Dibujar rectángulo aproximado (basado en cache)
                    if 'bbox' in mat_info:
                        x1, y1, x2, y2 = mat_info['bbox']
                        cv2.rectangle(annotated, (x1, y1), (x2, y2), (255, 0, 0), 2)
                    break
    
    # Asociar matrículas con vehículos y escribir en CSV
    matriculas_asociadas = set()
    
    for obj in objetos_trackeados:
        mejor_matricula = None
        
        # Si es un vehículo, buscar matrícula asociada
        if obj['tipo'] in ['car', 'motorcycle', 'bus', 'truck']:
            track_id = obj['track_id']
            
            # OPTIMIZACIÓN: Primero verificar si ya tenemos la matrícula en cache
            if track_id in matriculas_cache:
                mej

c:\Users\juanf\anaconda3\envs\VC_P4\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando modelo smolVLM...


c:\Users\juanf\anaconda3\envs\VC_P4\lib\site-packages\transformers\models\auto\modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Modelo cargado en: cuda
Iniciando procesamiento del video...
Optimizaciones activas:
  - OCR cada 5 frames
  - Cache de matrículas por tracking ID
  - Sin visualización en tiempo real

video 1/1 (frame 1/2832) c:\Users\juanf\Desktop\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 76.9ms
Procesando fotograma 0...
video 1/1 (frame 2/2832) c:\Users\juanf\Desktop\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 11.8ms
video 1/1 (frame 3/2832) c:\Users\juanf\Desktop\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 25.1ms
video 1/1 (frame 4/2832) c:\Users\juanf\Desktop\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 15.2ms
video 1/1 (frame 5/2832) c:\Users\juanf\Desktop\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 14.9ms
video 1/1 (frame 6/2832) c:\Users\juanf\Desktop\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 15.8ms
video 1/1 (frame 7/2832) c:\Users\juanf\Desktop\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 17.1ms
video 1/1 (frame 8/2832) c:\Users\juanf\Desktop\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 16.2ms
video 1/1 (fr

KeyboardInterrupt: 